<h1>RAZ Systems </h1>

Problem Statement

You are tasked with building a simple banking system simulator using Python and SQLite. The system should allow users to store and retrieve bank account information such as customer name and account balance.

In addition to basic database operations, you will design a smart assistant agent that can respond to user queries about bank accounts in natural language.

This notebook uses **AutoGen AgentChat** -- the high-level API, built on top of AutoGen Core. If you've used the OpenAI Agents SDK, the rough mapping is:

| OpenAI Agents SDK | AutoGen AgentChat |
|---|---|
| `Agent(instructions=..., model=...)` | `AssistantAgent(system_message=..., model_client=...)` |
| `Runner.run(agent, message)` | `await agent.on_messages([message], cancellation_token=...)` |
| a plain string prompt | a `TextMessage` object |

We'll build this up concept by concept, the same way the lesson notebooks do.

In [7]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### First concept: Select Which Model to use

`OpenAIChatCompletionClient` is AutoGen's wrapper around an OpenAI-compatible model -- this is the AgentChat equivalent of passing `model="gpt-4o-mini"` directly into `Agent(...)` in the OpenAI Agents SDK. Here, the model lives in its own object, because the *same* client can be reused across several agents.

In [8]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

### Second concept: Construct the Message

Unlike the Agents SDK, where you pass a plain string into `Runner.run(agent, "some prompt")`, AgentChat agents expect a structured message object. `TextMessage` carries the text *and* a `source` field (who said it), which matters once more than one agent or a human is in the conversation.

In [10]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="What is the balance for account ACC1003?", source="user")
message

TextMessage(id='91c6cfc1-98b4-448f-87bd-beb6e9a862df', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 28, 5, 6, 17, 482968, tzinfo=datetime.timezone.utc), content='What is the balance for account ACC1003?', type='TextMessage')

### Third concept: Build an Agent

This is the AgentChat equivalent of `Agent(name=..., instructions=..., model=...)`. Three differences worth noting:
- `system_message` here is the same idea as `instructions` in the Agents SDK
- `model_client_stream=True` enables token-by-token streaming, similar in spirit to `Runner.run_streamed()`



In [11]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="bank_teller_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful banking assistant. "
        "You provide short, professional answers about customer bank account balances."
    ),
    model_client_stream=True
)

### Fourth concept: Talk to the agent with `on_messages` -- but it has no data yet

`on_messages()` is the AgentChat method that actually runs the agent -- it's the rough equivalent of `Runner.run(agent, message)` in the Agents SDK, just plural (`[message]`, a list) and paired with a `CancellationToken` so a caller can cancel an in-flight call. It returns a `Response` object with a `.chat_message` (the final reply).

**This cell is not "putting it all together" yet -- it's the opposite.** The agent has no access to any real account data at this point, so watch it correctly refuse to make up a balance. That refusal is the whole reason we need a *tool* -- which is what the rest of this notebook builds.

In [12]:
from autogen_core import CancellationToken

response = await agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

"I'm sorry, but I cannot access specific account balances. Please check your account through your bank's online portal or contact customer service for assistance."

### Fifth concept: Give the agent real data -- a local SQLite database of bank accounts

We'll create a tiny `bank_accounts` table and populate it with a few sample accounts. This is the data the agent above couldn't see -- and it's what we'll expose to it as a tool next.

In [14]:
import os
import sqlite3

# Delete existing database file if it exists
if os.path.exists("bank.db"):
    os.remove("bank.db")

# Create the database and the table
conn = sqlite3.connect("bank.db")

c = conn.cursor()

c.execute("""
CREATE TABLE bank_accounts (
    account_number TEXT PRIMARY KEY,
    customer_name TEXT,
    balance REAL
)
""")

conn.commit()
conn.close()

In [15]:
# Populate our database
def save_account_balance(account_number, customer_name, balance):
    conn = sqlite3.connect("bank.db")
    c = conn.cursor()

    c.execute("""
    REPLACE INTO bank_accounts
    (account_number, customer_name, balance)
    VALUES (?, ?, ?)
    """, (account_number, customer_name, balance))

    conn.commit()
    conn.close()


# Sample bank accounts
save_account_balance("ACC1001", "Ajaz Pasha", 15000.75)
save_account_balance("ACC1002", "Aiza Khan", 24500.00)
save_account_balance("ACC1003", "Numan Khan", 8750.50)
save_account_balance("ACC1004", "Mohammed", 32000.25)
save_account_balance("ACC1005", "N Jahan", 12600.00)

In [16]:
def get_account_balance(account_number: str) ->str:
    conn = sqlite3.connect("bank.db")
    c = conn.cursor()

    c.execute("""
    SELECT customer_name, balance
    FROM bank_accounts
    WHERE account_number = ?
    """, (account_number,))

    result = c.fetchone()

    conn.close()

    if result:
        customer_name, balance = result
        return f"Customer: {customer_name}, Balance: ${balance}"
    else:
        return "Account not found"



In [17]:

# Example usage
print(get_account_balance("ACC1003"))

Customer: Numan Khan, Balance: $8750.5


### Sixth concept: Wire the database lookup in as a tool

`get_account_balance` is just a plain Python function -- AgentChat will inspect its type hints to build the tool schema for you, the same automatic behavior as `@function_tool` in the OpenAI Agents SDK (no separate decorator needed here; passing the function directly in `tools=[...]` is enough).

**`reflect_on_tool_use=True`** is the parameter worth paying attention to. Without it, the agent's final reply would just be the tool's raw return string (`"Customer: ..., Balance: $..."`). With it, the model takes one more pass over the tool's result and rephrases it into a natural reply -- the difference between a tool *returning* an answer and an agent *responding* with one.

In [18]:
from autogen_agentchat.agents import AssistantAgent

smart_bank_agent = AssistantAgent(
    name="smart_bank_agent",
    model_client=model_client,
    system_message=(
        "You are a helpful banking assistant. "
        "You provide short, professional answers about customer bank account balances."
    ),
    model_client_stream=True,
    tools=[get_account_balance],
    reflect_on_tool_use=True
)

In [19]:
response = await smart_bank_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

[FunctionCall(id='call_QgC6lGH5ps0X3hySswUlpMDM', arguments='{"account_number":"ACC1003"}', name='get_account_balance')]
[FunctionExecutionResult(content='Customer: Numan Khan, Balance: $8750.5', name='get_account_balance', call_id='call_QgC6lGH5ps0X3hySswUlpMDM', is_error=False)]


'The balance for account ACC1003 is $8,750.50.'

**What you're seeing above:** `response.inner_messages` shows the *intermediate* steps the agent took -- a `FunctionCall` (the model deciding to call `get_account_balance` with `account_number="ACC1003"`), then a `FunctionExecutionResult` (what the function actually returned). `response.chat_message.content` is the final, reflected-on natural-language answer.



In [20]:
message

TextMessage(id='91c6cfc1-98b4-448f-87bd-beb6e9a862df', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 28, 5, 6, 17, 482968, tzinfo=datetime.timezone.utc), content='What is the balance for account ACC1003?', type='TextMessage')

---
## What to try next

- Ask about an account number that doesn't exist (e.g. `ACC9999`) and see how the agent handles `"Account not found"`
- Set `reflect_on_tool_use=False` and compare the raw tool output to the reflected answer
- Add a second tool, e.g. `list_all_accounts()`, and ask a question that requires it instead
- Try `agent.run(task="...")` instead of `on_messages([...])` and compare the two call styles